# Supply Chain Cost Intelligence — Notebook 02: Supplier Clustering

**Goals:**
- Run SQL queries 01 and 03 via DuckDB to build the vendor feature table
- K-means clustering with elbow + silhouette analysis for optimal k
- Hierarchical clustering as comparison
- Assign **named business segments** (not 'Cluster 0')
- Quantify dollar savings opportunities for Underperforming segment
- Visualize clusters in 2D (PCA-reduced) Plotly scatter

**Key interview story:** Output is framed as business recommendations with dollar estimates, not model output.

In [ ]:
import sys; sys.path.insert(0, '..')
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.decomposition import PCA
from pathlib import Path
from src.data_loader import DB_PATH
from src.clustering import VendorSegmenter, build_vendor_features

sns.set_theme(style='darkgrid')
figures = Path('../figures')
conn = duckdb.connect(str(DB_PATH), read_only=True)
print('Setup complete')

## 1. Build Vendor Feature Table via SQL

Run `sql/01_vendor_performance.sql` to get the vendor-level KPI table.

In [ ]:
vendor_sql = open('../sql/01_vendor_performance.sql').read()
# Strip the COPY command from end — we'll save to parquet separately
vendor_sql_clean = vendor_sql.split('-- ── Save to parquet')[0]
vendor_df = conn.execute(vendor_sql_clean).df()
print(f'Vendor table: {vendor_df.shape}')
vendor_df.head()

## 2. Feature Inspection

Check distributions before scaling — K-means is sensitive to scale and outliers.

In [ ]:
vendor_df_feat, feat_cols = build_vendor_features(vendor_df)
vendor_df_feat[feat_cols].describe()

In [ ]:
# TODO: Box plots of each feature — identify outliers to clip before clustering
# Save to figures/02_feature_distributions.png

## 3. Elbow + Silhouette Analysis

Find optimal k. Silhouette score penalizes both too few and too many clusters.
Expect k=4 based on the four natural business tiers in the brief.

In [ ]:
segmenter = VendorSegmenter(max_k=10)
segmenter.fit(vendor_df)
print(f'Best k: {segmenter.best_k}')
print(f'Silhouette scores: {[round(s,3) for s in segmenter.silhouettes]}')

In [ ]:
k_range = range(2, len(segmenter.silhouettes) + 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_range, segmenter.inertias, 'o-')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow')
axes[1].plot(k_range, segmenter.silhouettes, 'o-', color='green')
axes[1].axvline(segmenter.best_k, color='red', linestyle='--', label=f'best k={segmenter.best_k}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('Silhouette')
axes[1].legend()
plt.tight_layout()
fig.savefig(figures / '02_elbow_silhouette.png', dpi=120)
plt.show()

## 4. Assign Segments and Name Clusters

In [ ]:
vendor_segmented = segmenter.assign_segments(vendor_df)
print(vendor_segmented['segment_name'].value_counts())

## 5. Cluster Visualization (2D PCA)

Reduce to 2 dimensions with PCA, color by segment name.
This is the chart that goes in the Quarto report.

In [ ]:
from sklearn.preprocessing import StandardScaler
feat_df, feat_cols = build_vendor_features(vendor_segmented)
X_scaled = StandardScaler().fit_transform(feat_df[feat_cols].fillna(0))
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
feat_df['pc1'] = coords[:, 0]
feat_df['pc2'] = coords[:, 1]
feat_df['segment_name'] = vendor_segmented['segment_name'].values
feat_df['recipient_name'] = vendor_segmented['recipient_name'].values
feat_df['lifetime_spend'] = vendor_segmented['lifetime_spend'].values

color_map = {
    'Premium Reliable': '#3ecf8e',
    'Cost-Efficient':   '#63b3ed',
    'Underperforming':  '#e53e3e',
    'Risky / Volatile': '#f5a623',
}
fig = px.scatter(
    feat_df, x='pc1', y='pc2',
    color='segment_name',
    color_discrete_map=color_map,
    size='lifetime_spend',
    size_max=30,
    hover_name='recipient_name',
    title='Supplier Segments (PCA projection)',
    labels={'pc1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            'pc2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)',},
    template='plotly_dark',
)
fig.write_html(str(figures / '02_cluster_scatter.html'))
fig.show()

## 6. Cost Reduction Opportunity Estimate

This is the number that goes in the README and on the resume bullet.

In [ ]:
savings_summary = segmenter.estimate_savings(vendor_segmented)
display(savings_summary)

## 7. Run SQL Opportunity Query

Run `sql/03_cost_opportunities.sql` for vendor-level opportunity drill-down.

In [ ]:
opp_sql = open('../sql/03_cost_opportunities.sql').read().split('-- ── Opportunity 2')[0]
opps = conn.execute(opp_sql).df()
total_conservative = opps['conservative_savings_30pct'].sum()
print(f'Total conservative savings opportunity: ${total_conservative:,.0f}')
display(opps.head(20))

## 8. Save Segmented Table

In [ ]:
from pathlib import Path
Path('../data/processed').mkdir(exist_ok=True)
vendor_segmented.to_parquet('../data/processed/vendor_segmented.parquet', index=False)
print('Saved vendor_segmented.parquet')

## Results Summary

| Metric | Value |
|--------|-------|
| Vendors clustered | TBD |
| Best k | TBD |
| Silhouette score | TBD |
| Premium Reliable (n) | TBD |
| Cost-Efficient (n) | TBD |
| Underperforming (n) | TBD |
| Risky/Volatile (n) | TBD |
| Conservative savings opportunity | $TBD |